# 04 – Fine-Tuning (Phase 5)

**NEEDS GPU** – T4×2 or P100. Enable GPU before running.

**Phase 5 exit gate:** Fine-tuned model compared against Phase 4 best on the same validation metric across ≥3 seeds. If it does not clearly win, it still may add value in ensemble.

**Read AGENTS.md Phase 5 recipe before running.** Stage A (head only) → Stage B (LLRD partial unfreeze) → Stage C (optional full FT).

In [ ]:
import sys, pathlib
import numpy as np
import torch

sys.path.insert(0, "..")
sys.path.append("/kaggle/input/datasets/somyajitsaha/food-retrieval-src")

from src.config import DATA_DIR, OUT_DIR, seed_everything, load_config
seed_everything(42)

import torchvision, timm, sklearn
try:
    import open_clip
    print("open_clip:", open_clip.__version__)
except ImportError:
    raise ImportError("open_clip required: pip install open-clip-torch")

print({k: v.__version__ for k, v in dict(
    torch=torch, torchvision=torchvision, timm=timm, sklearn=sklearn, numpy=np).items()})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else "N/A")

In [ ]:
import pandas as pd
from src.data import load_csvs, GalleryDataset, get_transforms, image_dir, parse_val_ground_truth
from src.models import load_clip_model, param_groups_lrd
from src.train import fine_tune, get_criterion, EMA
from src.features import extract_image_features, save_features, load_features, load_text_features
from src.metrics import evaluate, print_metrics
from torch.utils.data import DataLoader

dfs = load_csvs(DATA_DIR)
meta        = dfs["category_metadata"]
train_df    = dfs["train"]
val_gallery = dfs["val_gallery"]
val_queries = dfs["val_queries"]
test_gallery= dfs["test_gallery"]
test_queries= dfs["test_queries"]
val_gt      = parse_val_ground_truth(dfs["val_gt"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
K = meta["category_id"].nunique()
cat_id_list = meta["category_id"].tolist()
cat_id2idx  = {int(c): i for i, c in enumerate(cat_id_list)}

## Stage A – Head-only training (frozen backbone)

In [ ]:
# Sanity check: overfit 1 batch before long run (AGENTS.md Hard Rule 20)
from src.config import seed_everything
seed_everything(42)

MODEL_NAME  = "ViT-L-14"
PRETRAINED  = "openai"
IMG_SIZE    = 224
BATCH_SIZE  = 64
HEAD_LR     = 1e-3
WD          = 0.05
EPOCHS_A    = 5

model, preprocess_val, tokenize = load_clip_model(MODEL_NAME, PRETRAINED, DEVICE)

# Freeze backbone, only train linear head
for name, p in model.named_parameters():
    p.requires_grad = name.startswith("head") or name.startswith("fc")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable / 1e6:.2f}M")

tf_train = get_transforms("train", size=IMG_SIZE,
                           mean=(0.48145466, 0.4578275, 0.40821073),
                           std=(0.26862954, 0.26130258, 0.27577711))

id2cat  = dict(zip(train_df["image_id"], train_df["category_id"]))

# TODO: build a classification dataset (GalleryDataset returns (img, id); 
# wrap to return (img, cat_idx)) then call fine_tune()
print("TODO: wire training dataset and call fine_tune()")
print("See src/train.py for the fine_tune() API.")

## Stage B – Partial unfreeze with layer-wise LR decay

Run after Stage A converges. Load the Stage A checkpoint and unfreeze the last N blocks.

In [ ]:
# Example LLRD param groups (uncomment and adapt after Stage A)
# N_UNFROZEN = 4
# for name, p in model.named_parameters():
#     block_idx = int(name.split('blocks.')[1].split('.')[0]) if 'blocks.' in name else -1
#     p.requires_grad = (block_idx >= N_BLOCKS - N_UNFROZEN) or name.startswith('head')
#
# param_groups = param_groups_lrd(model, base_lr=1e-5, head_lr=1e-3, wd=0.05, decay=0.75)
# optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.999), eps=1e-8)
print("Stage B: implement after Stage A checkpoint is available.")

## Extract and cache fine-tuned features

In [ ]:
# After fine-tuning, load best EMA checkpoint and re-extract features
CKPT_TAG = "vitl14_openai_ft"
# ckpt = torch.load(pathlib.Path(OUT_DIR) / 'checkpoints' / 'finetune_vitl14_openai_best.pt')
# model.load_state_dict(ckpt['model_state_dict'])
# model.eval()
#
# for split, df in [('train', train_df), ('val', val_gallery), ('test', test_gallery)]:
#     ...
#     feats, ids = extract_image_features(model, loader, device=DEVICE, use_tta=True)
#     save_features(feats, ids, pathlib.Path(OUT_DIR)/'features'/CKPT_TAG, split)
print("TODO: run extraction after fine-tuning completes.")